# Figure PS03: does residual length explain the lower 3Di entropy of annotated phage ORFs?

**Status:** draft, Gadi (agent-handoffs#38). Supports Results, "Phage genes sit lower in 3Di entropy than cellular genes".

## Provenance

| plotting file | built from |
|---|---|
| `data/figPS03_length_hist.tsv.gz` | per-length counts: stop-to-stop ORF length of positives, length of every annotated CDS, and length of the CDS that ends at each positive's stop |
| `data/figPS03_gap_by_length.tsv.gz` | per exact aa length (90–999), n and mean 3Di entropy of positives, for four restriction sets |
| `data/tabPS03_length_matched_gap.tsv` | raw and exact-length-standardised bacteria − phage gaps, grouped-jackknife SE |
| `data/tabPS03_length_summary.tsv`, `tabPS03_percentile.tsv`, `tabPS03_states.tsv`, `figPS03_state_usage.tsv` | length summaries; length-conditioned percentiles; distinct states, Miller–Madow entropy and 3Di state usage |
| `data/figPS03_provenance.json` | seed, jackknife groups, sampled GTDB chunks, restriction sets |

- **Reduction:** `manuscript/prep/phage_ps03_prepare_data.py`, run as `prep/phage_ps03.pbs` (PBS 182890402[], 182890403, 182890404[], 182890405).
- **Phage:** Gadi Inphared run, 36,025 genomes, 2,685,382 positives (`in_genbank`), `genome_entropy` 0.3.0, CDS from the same Pyrodigal-gv/phold GenBank records.
- **Bacteria:** all 760 GTDB r232 chunks, 309,065,661 positives, deposited CDS from `gtdb_entropy/cds_intervals`. The per-length counts and means equal the published Fig. P04 bacterial matched curve at all 910 lengths (max |Δmean| 5 × 10⁻⁶ bits).
- **CDS cover:** the longest same-strand CDS ending at the ORF's stop, divided by ORF length (as `Validation/Inphared/scripts/label_restrict_gap.py`), now applied to both populations. **Truncated:** ORF within 3 nt of a record end.
- **Length matching:** exact amino-acid length, 90–999 aa; strata with ≥ 5 ORFs on both sides (100% of phage positives fall in the common support). The standardised gap weights each length by the phage count (bacterial weights also reported).
- **Uncertainty:** grouped delete-one jackknife; 760 GTDB chunks and 200 random phage genome groups (seed 0); the two variances are added. Phage near-duplicates are not grouped by ANI, so the phage SE is a lower bound.
- **3Di strings:** all phage positives and the positives of 10 GTDB chunks drawn with seed 0 (1,227 genomes, 3.85 M positives).


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker

DATA = Path("../data")
IMAGES = Path("../images")
IMAGES.mkdir(exist_ok=True)
COL = {"phage": "#CC79A7", "bacteria": "#0072B2"}
INK = "#222222"
MUTED = "#555555"
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 7.5,
    "axes.titlesize": 8, "axes.labelsize": 7.5,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": INK, "axes.labelcolor": INK,
    "xtick.color": INK, "ytick.color": INK,
    "svg.hashsalt": "genome_entropy", "pdf.fonttype": 42,
})


def panel(ax, letter):
    ax.text(-0.02, 1.04, letter, transform=ax.transAxes, fontsize=10,
            fontweight="bold", ha="right", va="bottom")


def save(fig, stem):
    fig.savefig(IMAGES / f"{stem}.pdf", bbox_inches="tight",
                metadata={"CreationDate": None, "ModDate": None,
                          "Producer": None, "Creator": None})
    fig.savefig(IMAGES / f"{stem}.png", dpi=300, bbox_inches="tight",
                metadata={"Software": None})
    print("wrote", IMAGES / f"{stem}.pdf", IMAGES / f"{stem}.png")


H = pd.read_csv(DATA / "figPS03_length_hist.tsv.gz", sep="\t")
C = pd.read_csv(DATA / "figPS03_gap_by_length.tsv.gz", sep="\t")
T = pd.read_csv(DATA / "tabPS03_length_matched_gap.tsv", sep="\t")
U = pd.read_csv(DATA / "figPS03_state_usage.tsv", sep="\t")
BINS = ["90-149", "150-299", "300-599", "600-999"]
SETS = {"all": "all positives (published)",
        "phage_cov80": "phage CDS ≥ 80% of ORF",
        "both_cov80_untrunc": "both ≥ 80%, untruncated"}


In [ ]:
fig = plt.figure(figsize=(7.2, 6.4))
gs = fig.add_gridspec(2, 2, hspace=0.8, wspace=0.32)

# a: length distributions
ax = fig.add_subplot(gs[0, 0])
EDGES = np.unique(np.round(np.geomspace(30, 3000, 61)).astype(int))
for (kind, ds), g in H.groupby(["kind", "dataset"]):
    if kind == "cds_of_positive":
        continue
    g = g[g.aa_length < 3000]
    if kind == "orf_positive":
        g = g[g.aa_length <= 999]   # >= 1,000 aa are pooled in the source; not shown
    h, _ = np.histogram(g.aa_length, EDGES, weights=g.n)
    dens = h / h.sum() / np.diff(np.log10(EDGES))
    x = np.sqrt(EDGES[:-1] * EDGES[1:])
    ls = "-" if kind == "orf_positive" else "--"
    lab = f"{ds}, {'matched ORF (stop-to-stop)' if kind == 'orf_positive' else 'every annotated CDS'}"
    ax.plot(x, dens, color=COL[ds], ls=ls, lw=1.0, label=lab)
ax.axvline(90, color=MUTED, lw=0.5, ls=(0, (1, 2)))
ax.set_xscale("log")
ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
ax.set_xlabel("length (aa)")
ax.set_ylabel("density per log₁₀ aa")
ax.set_xlim(30, 1000)
ax.set_xticks([30, 90, 300, 1000], ["30", "90", "300", "1,000"])
ax.legend(frameon=False, fontsize=5.5, loc="lower left", bbox_to_anchor=(0, 1.0), ncol=1)
panel(ax, "a")

# b: exact-length-matched gap within 40 log-spaced strata (as Fig. P04)
LB = np.unique(np.round(np.geomspace(90, 1000, 41)).astype(int))
ax = fig.add_subplot(gs[0, 1])
for s, ls in (("all", "-"), ("phage_cov80", ":"), ("both_cov80_untrunc", "--")):
    t = C[C.set == s].copy()
    t = t[t.n_phage > 0].copy()
    t["b"] = pd.cut(t.aa_length, LB, right=False)
    t["wl"] = t.n_phage * np.log(t.aa_length)
    # within each stratum, average the exact-length gaps with phage weights
    t["d"] = (t.mean_bacteria - t.mean_phage) * t.n_phage
    g = t.groupby("b", observed=True)[["n_phage", "wl", "d"]].sum()
    ax.plot(np.exp(g.wl / g.n_phage), g.d / g.n_phage, color=INK, ls=ls, lw=1.0, marker="o",
            ms=1.4, label=SETS[s])
ax.axhline(0, color=MUTED, lw=0.5)
ax.set_xscale("log")
ax.set_xlim(90, 1000)
ax.set_xticks([90, 150, 300, 600, 999], ["90", "150", "300", "600", "999"])
ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
ax.set_xlabel("ORF length (aa)")
ax.set_ylabel("bacteria − phage\nmean 3Di entropy (bits)")
ax.set_ylim(-0.05, 0.6)
ax.legend(frameon=False, fontsize=6, loc="upper right")
panel(ax, "b")

# c: raw vs exact-length-matched gaps by published bin
ax = fig.add_subplot(gs[1, 0])
w = 0.13
for k, s in enumerate(SETS):
    t = T[T.set == s].set_index("bin").loc[BINS]
    x = np.arange(len(BINS)) + (k - 1) * 2.2 * w
    ax.bar(x - w / 2, t.gap_raw, w, color="#BBBBBB", edgecolor=INK, lw=0.4,
           yerr=1.96 * t.se_raw, error_kw=dict(lw=0.5, capsize=0))
    ax.bar(x + w / 2, t.gap_matched_phage_w, w, color=["#999999", "#E69F00", "#D55E00"][k],
           edgecolor=INK, lw=0.4, yerr=1.96 * t.se_matched_phage_w,
           error_kw=dict(lw=0.5, capsize=0), label=f"{SETS[s]}: length-matched")
from matplotlib.patches import Patch
hs, ls_ = ax.get_legend_handles_labels()
hs.append(Patch(facecolor="#BBBBBB", edgecolor=INK, lw=0.4))
ls_.append("raw gap of bin means (left of each pair)")
ax.set_xticks(range(len(BINS)), [b.replace("-", "–") for b in BINS])
ax.set_xlabel("ORF length (aa)")
ax.set_ylabel("bacteria − phage (bits)")
ax.legend(hs, ls_, frameon=False, fontsize=5.5, loc="lower left", bbox_to_anchor=(0, 1.02), ncol=1)
panel(ax, "c")

# d: 3Di state usage, both >= 80%, untruncated, 90-999 aa
ax = fig.add_subplot(gs[1, 1])
u = U[U.bin == "90-999"].assign(d=lambda x: 100 * (x.freq_phage - x.freq_bacteria)).sort_values("d")
ax.barh(u.state, u.d, color=np.where(u.d > 0, COL["phage"], COL["bacteria"]), height=0.7)
ax.axvline(0, color=INK, lw=0.5)
ax.set_xlabel("phage − bacteria 3Di state frequency\n(percentage points)")
ax.set_ylabel("3Di state")
ax.tick_params(axis="y", labelsize=6)
panel(ax, "d")
save(fig, "figPS03_phage_length_matched")


In [ ]:
pd.set_option("display.width", 200)
cols = ["set", "bin", "n_bacteria", "n_phage", "gap_raw", "se_raw", "gap_matched_phage_w",
        "se_matched_phage_w", "gap_matched_bacteria_w", "explained_by_length"]
print(T[cols].round(4).to_string(index=False))
print(pd.read_csv(DATA / "tabPS03_length_summary.tsv", sep="\t").round(1).to_string(index=False))
print(pd.read_csv(DATA / "tabPS03_percentile.tsv", sep="\t").round(3).to_string(index=False))
S = pd.read_csv(DATA / "tabPS03_states.tsv", sep="\t")
print(S[["set", "bin", "n_phage", "n_bacteria", "h_plugin_gap_matched", "h_mm_gap_matched",
         "bias_term_gap_matched", "k_distinct_gap_matched"]].round(4).to_string(index=False))


## Caption (draft)

**Figure PS03. Residual length explains little of the lower 3Di entropy of annotated phage ORFs.**

**(a)** Length distributions. Solid lines: stop-to-stop ORFs matched to an annotated CDS (the ORFs whose entropy is measured, ≥ 90 aa). Dashed lines: every annotated CDS. ORFs of 1,000 aa and longer are not shown. Annotated phage CDS are much shorter than bacterial ones (median 135 against 274 aa), but the 90-aa floor removes most of that difference: matched phage ORFs have median 184 aa against 323 aa overall, and within each published length bin the medians differ by at most 26 aa.

**(b)** Bacteria-minus-phage mean 3Di entropy, matched on exact length and averaged with phage weights within 40 log-spaced strata, for all positives, for phage positives whose CDS covers ≥ 80% of the ORF, and for positives of both populations with ≥ 80% cover and no truncation.

**(c)** Gaps in the four published bins: raw bin means (grey) and exact-length-standardised gaps weighted by the phage length distribution (coloured), with 95% grouped-jackknife intervals. Exact length matching removes 15% of the gap at 90–149 and 150–299 aa and none above 300 aa. Restricting both populations to ≥ 80% cover **widens** the gap, to 0.26–0.30 bits in every bin.

**(d)** 3Di state usage of positives with ≥ 80% cover and no truncation (all phage positives; bacterial positives of 1,227 genomes in 10 random GTDB chunks). Phage ORFs use more D, V and P and fewer of most other states.
